In [1]:
import sys
import random
import time
from datetime import datetime
from glob import glob
import warnings
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from pandas import DataFrame
from scipy.stats import genextreme
from numpy import array, mean, min, max, sqrt, log, arange
from scipy.stats import norm
import multiprocessing as mp
from joblib import Parallel, delayed
from tqdm import tqdm

import func_gev as gev
import func_preparation as dbf
import func_plotting as dbplt
import func_utils as ut

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

# Settings

In [9]:
path_input = '../input/Annual_max_DCPP_20260112/'
path_export = '../output/gev_analysis/pooled/'

In [3]:
hindcast_start = 1960
hindcast_end = 2026

In [4]:
# adding a little randomness to enhance trust the model works as robust as possible cross locations..
start_location = random.choice(arange(0, 9579))
end_location = start_location+10

print(f'analyse a subsample of location {start_location}–{end_location}')

analyse a subsample of location 7253–7263


In [5]:
return_periods = [10, 25, 50, 100, 200]
plot_period_evolution = ['10-year', '50-year', '100-year']

In [6]:
colors = [
    '#53354DFF','#7D4F73FF','#B887ADFF','#CAA5C2FF','#DBC3D6FF','#F5F5F5FF','#99E3DDFF',
    '#66D4CCFF','#33C6BBFF','#008A80FF','#005C55FF'
    ]

In [7]:
_LOCATION_LABELS = None

In [8]:
print_msg = True
export_report=True
display_results = False

# Import Data

In [16]:
ls_files = [file for file in glob(path_input + '*.nc')]

print('Importing Data from ...')
print("\n".join(ls_files))

dic_data_per_model = dbf.import_all_models(ls_files)


Importing Data from ...
../input/Annual_max_DCPP_20260112/Annual_max_MIROC6.nc
../input/Annual_max_DCPP_20260112/Annual_max_MPI-ESM1-2-HR.nc
../input/Annual_max_DCPP_20260112/Annual_max_HadGEM3-GC31-MM.nc
../input/Annual_max_DCPP_20260112/Annual_max_MRI-ESM2-0.nc
../input/Annual_max_DCPP_20260112/Annual_max_BCC-CSM2-MR.nc
../input/Annual_max_DCPP_20260112/Annual_max_CMCC-CM2-SR5.nc
../input/Annual_max_DCPP_20260112/Annual_max_CanESM5.nc
../input/Annual_max_DCPP_20260112/Annual_max_NorCPM1.nc


# Prepare Data

### Pooling, BiasCorrection, ValidityCheck

In [18]:
print('Pooling and Preparing Data...')
dic_data_per_model, combined, notes_overview = dbf.prepare_combined_data(ls_files, dic_data_per_model)
print('... done.')

Pooling and Preparing Data...


/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


... done.


### Rearrangement to per location

In [ ]:
print('Rearranging Data – sorting per location...')    
dic_data_per_location = dbf.extract_location_data(combined, hindcast_start, hindcast_end)

Rearranging Data – sorting per location...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    7.9s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    8.1s
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:    8.2s
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:    8.3s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    8.4s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.19597432649575183s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:    8.5s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:    8.6s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.16751313209533691s.) Setting batch_size=4.
[Parallel(n_jobs=-1)]: Done  82 tasks      | elapsed:    8.8s
[Parallel(n_jobs=-1)]: Done 128 tasks      | elapsed:    9.0s
[Parallel(n_jobs=-1)]: Done 188 tasks      | elapsed:    9.3s
[Parallel(n_jobs=-1)]: Done 248 tasks      | elapsed:    9.5s
[Parallel(n_jobs=-1)]: Done 316 tasks      | elaps

## Select Subset

In [ ]:
if start_location is not None or end_location is not None: 
    dic_data_per_location = ut.select_allowed_locations(
        dic_data_per_location=dic_data_per_location, 
        start_loc=start_location, end_loc=end_location
        )
    print(f'Processing locations {start_location} to {end_location} ({len(dic_data_per_location)} total)')

else:
    print(f'Processing all {len(dic_data_per_location)} locations')

print('Getting closest point available as location label for orientation. \nNote this is not the exact location...')
location_labels = dbf.precompute_location_labels(dic_data_per_location)
location_labels

# Non-Stationary GEV Analysis

In [ ]:
output, ls_notes = run_gev_parallel(dic_data_per_location, location_labels)  
results = output['results']  

# ut.save_pooled_results(results=results, data=output['data'], base_dir=path_child_folder)